# Stateful agents, persistence, and memory with LangGraph

## Northstar Cloud incident investigator

This notebook is a complete, credential-free practical lesson. You will model an investigation as explicit state and graph-shaped nodes; checkpoint every durable boundary; recover after a simulated worker loss; pause for human approval; and keep unverified cross-thread memory from biasing a diagnosis.

**Scenario.** European checkout failures began after `deploy-1842`. Health is mostly green, so an agent must gather independent evidence before it proposes (never executes) a rollback.

### Outcomes

- Separate thread-scoped state/checkpoints from governed long-term memory.
- Build bounded conditional routing and inspect its trajectory.
- Resume from a checkpoint without repeating completed work.
- Model an approval pause safely and understand LangGraph `interrupt()` semantics.
- Translate the dependency-free model into a real `StateGraph`.


## Execution lifecycle

![Stateful LangGraph incident investigation lifecycle](assets/langgraph-state-memory-lifecycle.svg)

The SVG is a repository asset rather than a browser-loaded Mermaid diagram so it renders on GitHub, nbviewer, and local Jupyter without JavaScript. The course README contains the editable Mermaid versions.


## 1. The design contract

A stateful agent is not a conversation buffer with extra tools. It has an explicit state schema, deterministic nodes, conditional edges, durable snapshots, and controlled side effects. LangGraph is suited to this because it lets a graph mix hand-coded routing and validation with model-driven reasoning.

For this incident, **state** contains the current request, evidence, hypothesis, confidence, attempt budget, approval status, and safe events. Do not put hidden reasoning, unlimited raw logs, credentials, or unrelated user history into it.

A loop is allowed only while `confidence < 0.80` and the evidence budget remains. That turns an open-ended agent into an auditable state machine.


In [ ]:
from lab import (
    Checkpointer, MemoryStore, run_investigation, resume_investigation,
    resume_approval, stream
)

store = MemoryStore()
checkpointer = Checkpointer()
store.write(("customer", "acme"), {
    "kind": "preference",
    "value": "Prioritize clear impact updates.",
    "verified": True,
})
# A historical hunch is deliberately not trusted as evidence.
store.write(("customer", "acme"), {
    "kind": "diagnostic_hunch",
    "value": "Checkout problems are usually Redis.",
    "verified": False,
)


## 2. Run a bounded evidence loop

`lab.py` intentionally uses deterministic fake tools. This makes the trace repeatable: health → EU logs → deployment history. A production model can decide *which* authorized evidence source to query, but routing thresholds, rate limits, permissions, and final actions should stay deterministic.


In [ ]:
state = run_investigation(
    "Why are European customers failing checkout?",
    checkpointer,
    store,
    thread_id="incident-eu-1842",
)
print("status:", state.status)
print("hypothesis:", state.hypothesis)
print("recommendation:", state.recommendation)
print("checkpoint nodes:", checkpointer.history(state.thread_id))
for event in stream(state):
    print(f"- {event['node']}: {event['message']}")


## 3. Checkpoints: recover, do not restart

A LangGraph checkpointer stores graph-state snapshots for one `thread_id`. That is short-term, thread-scoped memory. It enables recovery, review, and an approval pause. It is *not* a cross-user profile.

Experiment: simulate losing the worker immediately after the evidence checkpoint. The snapshot already includes completed work, so the resumed run continues rather than recreating a fresh investigation. In production, use a durable saver and configure retention; an in-memory saver disappears on restart.


In [ ]:
recovery_store, recovery_checkpointer = MemoryStore(), Checkpointer()
try:
    run_investigation(
        "Investigate EU checkout failures", recovery_checkpointer, recovery_store,
        thread_id="recovery-demo", fail_after="collect_evidence"
    )
except RuntimeError as exc:
    print("simulated failure:", exc)

print("saved trajectory:", recovery_checkpointer.history("recovery-demo"))
resumed = resume_investigation(recovery_checkpointer, "recovery-demo")
print("resumed status:", resumed.status)
print("resumed evidence sources:", [item['source'] for item in resumed.evidence])


## 4. Long-term memory is a governed store

LangGraph’s two persistence primitives have distinct jobs:

| Primitive | Scope | Appropriate content |
| --- | --- | --- |
| **Checkpointer** | one graph thread | current evidence, task state, interrupt cursor |
| **Store** | across threads | verified user preferences, approved knowledge, shared facts |

A memory write needs provenance, a tenant/purpose namespace, verification, retention, deletion, and an authorized retrieval path. The lab’s `read_verified()` rejects the stale Redis hunch. This is a safety property: memory that is plausible but unsupported should not influence the diagnosis.

**Challenge:** temporarily inspect the private store in a debugger and explain why simply adding every item to model context would create both a reliability and a tenant-isolation risk. Do not change the production retrieval gate.


## 5. Interrupt before a consequential action

The lab ends in `paused`: it has prepared a rollback proposal but not executed it. A real graph can call `interrupt()` from a node, persist the state, and later resume with `Command(resume=...)` using the **same** `thread_id`. Code before an interrupt can run again after resume, so mutations before the pause must be idempotent.


In [ ]:
print("approval required?", state.pending_approval)
resume_approval(state, approved=False)  # Reject: no production side effect occurs.
checkpointer.save(state, "end")
print("final status:", state.status)
print("approved:", state.approved)
print("last event:", state.events[-1])


## 6. Real LangGraph translation

Install `langgraph` to use the following shape in an application. The exact model and tool integrations are intentionally omitted: this module focuses on orchestration, persistence, and memory boundaries.

```python
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command, interrupt

builder = StateGraph(IncidentState)
builder.add_node("triage", triage)
builder.add_node("collect_evidence", collect_evidence)
builder.add_node("analyze", analyze)
builder.add_node("approval", approval_node)
builder.add_edge(START, "triage")
builder.add_edge("triage", "collect_evidence")
builder.add_conditional_edges("analyze", route_after_analysis,
    {"collect_evidence": "collect_evidence", "approve": "approval"})
builder.add_edge("approval", END)
graph = builder.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

config = {"configurable": {"thread_id": "opaque-uuid"}}
# On interrupt: graph.invoke(Command(resume={"approved": True}), config=config)
```

For production replace in-memory persistence, authorize the user before any store access or approval, keep tool side effects idempotent, redact streamed state, and trace node-level outcomes.


## 7. Production readiness and exercises

Before deployment, add a maximum node count and deadline; typed reducers that prevent duplicate evidence; tool permission checks; retry classes; durable checkpoint retention; encrypted and namespaced stores; deletion workflows; trace redaction; and trajectory-level evaluations.

1. Add a `needs_human_review` branch for contradictory evidence.
2. Add a source-id reducer so replayed evidence cannot increase confidence.
3. Write a test that a `tenant-b` item cannot be retrieved in an Acme thread.
4. Stream only node name, elapsed time, and safe status to a UI.
5. Compare a fixed workflow against a model-routed graph on the same incident, then measure trajectory length, recovery correctness, and cost.

## References

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [Persistence: checkpointers and stores](https://docs.langchain.com/oss/python/langgraph/persistence)
- [Memory concepts](https://docs.langchain.com/oss/python/concepts/memory)
- [Interrupts and resume semantics](https://docs.langchain.com/oss/python/langgraph/interrupts)
- [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560)
